<a href="https://colab.research.google.com/github/hadri1-bisch/Advanced-Data-Analysis/blob/main/scm_survey_part_Beta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Modélisation du questionnaire:choix entre un pull neuf et un pull de seconde main**

NB : le produit choisi est un pull. Les prix ont été adaptés à ce produit :
- Neuf : 40, 60, 80 CHF (fast fashion, marque moyenne, bonne marque)
- Seconde main : 20, 40 CHF (particulier, magasin professionnel)

Nous sommes partis du principe qu'un prix plus élevé pour la seconde main ne ferait pas de sens sauf si c'est un pull de marque. Comme ici on veut tester les répondants sur un pull basique nous préférons rester sur des prix auxquels nous pensons que des jeunes serait prêt à payer un pull de seconde main.

**A) créations des profils**

In [1]:
import itertools
import pandas as pd
import numpy as np


In [2]:
# Définition des attributs PAR ALTERNATIVE (design labellisé : Neuf vs Seconde main)

# ---- Alternative "Neuf" : 3 attributs ----
prix_neuf_niveaux = [40, 60, 80]
unicite_neuf_niveaux = [1, 2]   # 1: Standard de collection courante, 2: Pièce unique/vintage introuvable en magasin
garantie_neuf_niveaux = [1, 2]     # 1: Sans retour, 2: Retours 30j

profils_neuf = list(itertools.product(
    prix_neuf_niveaux, unicite_neuf_niveaux, garantie_neuf_niveaux
))
df_profils_neuf = pd.DataFrame(
    profils_neuf, columns=['Prix', 'Unicité', 'Garantie']
)

# ---- Alternative "Seconde main" : 4 attributs (Qualité est spécifique à la SM) ----
prix_sm_niveaux = [20, 40]
qualite_sm_niveaux = [1, 2]        # 1: Bon état avec des traces d'usure visibles, 2: Comme neuf
unicite_sm_niveaux = [1, 2]    # 1: Standard de collection courante, 2: Pièce unique/vintage introuvable en magasin
garantie_sm_niveaux = [1, 2]       # 1: Sans retour, 2: Retours 30j

profils_sm = list(itertools.product(
    prix_sm_niveaux, qualite_sm_niveaux, unicite_sm_niveaux, garantie_sm_niveaux
))
df_profils_sm = pd.DataFrame(
    profils_sm, columns=['Prix', 'Qualité', 'Unicité', 'Garantie']
)

print(f"Profils Neuf (plein factoriel) : {len(df_profils_neuf)}")
print(df_profils_neuf.to_string(index=False))
print()
print(f"Profils Seconde main (plein factoriel) : {len(df_profils_sm)}")
print(df_profils_sm.to_string(index=False))

Profils Neuf (plein factoriel) : 12
 Prix  Unicité  Garantie
   40        1         1
   40        1         2
   40        2         1
   40        2         2
   60        1         1
   60        1         2
   60        2         1
   60        2         2
   80        1         1
   80        1         2
   80        2         1
   80        2         2

Profils Seconde main (plein factoriel) : 16
 Prix  Qualité  Unicité  Garantie
   20        1        1         1
   20        1        1         2
   20        1        2         1
   20        1        2         2
   20        2        1         1
   20        2        1         2
   20        2        2         1
   20        2        2         2
   40        1        1         1
   40        1        1         2
   40        1        2         1
   40        1        2         2
   40        2        1         1
   40        2        1         2
   40        2        2         1
   40        2        2         2


B) **Filtrage des profils iréalsites**

Neuf : 3 attributs
- Prix : 40, 60, 80 CHF
- Unicité : 1 (Courant), 2 (Pièce unique)
- Garantie : 1 (Sans retour), 2 (Retours 30j)

Seconde main : 4 attributs
- Prix : 20, 40 CHF
- Qualité : 1 (Bon état, traces d'usure visibles), 2 (Comme neuf)
- Unicité : 1 (Courant), 2 (Pièce unique)
- Garantie : 1 (Sans retour), 2 (Retours 30j)

In [3]:
print(f"Profils Neufs : {len(df_profils_neuf)}")
print(df_profils_neuf.to_string(index=False))
print()
print(f"Profils Seconde main : {len(df_profils_sm)}")
print(df_profils_sm.to_string(index=False))

Profils Neufs : 12
 Prix  Unicité  Garantie
   40        1         1
   40        1         2
   40        2         1
   40        2         2
   60        1         1
   60        1         2
   60        2         1
   60        2         2
   80        1         1
   80        1         2
   80        2         1
   80        2         2

Profils Seconde main : 16
 Prix  Qualité  Unicité  Garantie
   20        1        1         1
   20        1        1         2
   20        1        2         1
   20        1        2         2
   20        2        1         1
   20        2        1         2
   20        2        2         1
   20        2        2         2
   40        1        1         1
   40        1        1         2
   40        1        2         1
   40        1        2         2
   40        2        1         1
   40        2        1         2
   40        2        2         1
   40        2        2         2


**C) Question a insérer dans le questionnaires**

créations de toutes les paires candidates:

In [4]:
from itertools import product

In [5]:
# Générer toutes les paires : seconde main vs neuf
paires = []

for (idx_sm, profil_sm), (idx_neuf, profil_neuf) in product(
    df_profils_sm.iterrows(),
    df_profils_neuf.iterrows()
):
    paires.append({
        "Prix_SM": profil_sm["Prix"],
        "Qualite_SM": profil_sm["Qualité"],
        "Unicite_SM": profil_sm["Unicité"],
        "Retours_SM": profil_sm["Garantie"],
        "Prix_neuf": profil_neuf["Prix"],
        "Unicite_neuf": profil_neuf["Unicité"],
        "Retours_neuf": profil_neuf["Garantie"]
    })

df_paires_candidates = pd.DataFrame(paires)

print(f"Nombre de paires candidates : {len(df_paires_candidates)}")
print(df_paires_candidates.to_string(index=False))


Nombre de paires candidates : 192
 Prix_SM  Qualite_SM  Unicite_SM  Retours_SM  Prix_neuf  Unicite_neuf  Retours_neuf
      20           1           1           1         40             1             1
      20           1           1           1         40             1             2
      20           1           1           1         40             2             1
      20           1           1           1         40             2             2
      20           1           1           1         60             1             1
      20           1           1           1         60             1             2
      20           1           1           1         60             2             1
      20           1           1           1         60             2             2
      20           1           1           1         80             1             1
      20           1           1           1         80             1             2
      20           1           1          

In [6]:
# Supprimer les questions évidentes :
# une option est au moins aussi bonne sur le prix, l'unicité ET les retours
def est_evident(row):
    sm_mieux = (row["Prix_SM"] <= row["Prix_neuf"]
                and row["Unicite_SM"] >= row["Unicite_neuf"]
                and row["Retours_SM"] >= row["Retours_neuf"]
                )
    neuf_mieux = (row["Prix_neuf"] <= row["Prix_SM"]
                  and row["Unicite_neuf"] >= row["Unicite_SM"]
                  and row["Retours_neuf"] >= row["Retours_SM"])
    return sm_mieux or neuf_mieux

df_paires_candidates = (
    df_paires_candidates[~df_paires_candidates.apply(est_evident, axis=1)]
    .reset_index(drop=True)
)
print("Paires restantes :", len(df_paires_candidates))

Paires restantes : 74


**Choix des questions**
but min d-error

In [7]:
pip install choicedesign

In [8]:
import choicedesign as cd

Design D-efficient avec priors supposéss (pull)
- Prix linéaire : un seul coefficient par franc
- Retours séparés : un effet pour la seconde main, un pour le neuf
- Priors supposés (signe attendu), à remplacer par les estimations du questionnaire test

In [9]:
from choicedesign.design import EffDesign
from choicedesign.expressions import Attribute, Parameter, ASC

# --------------------------------------------------
# 1. Attributs
# --------------------------------------------------
prix_sm = Attribute("Prix_SM", prix_sm_niveaux)
qualite_sm = Attribute("Qualite_SM", qualite_sm_niveaux)
unicite_sm = Attribute("Unicite_SM", unicite_sm_niveaux)
retours_sm = Attribute("Retours_SM", garantie_sm_niveaux)

prix_neuf = Attribute("Prix_neuf", prix_neuf_niveaux)
unicite_neuf = Attribute("Unicite_neuf", unicite_neuf_niveaux)
retours_neuf = Attribute("Retours_neuf", garantie_neuf_niveaux)

# --------------------------------------------------
# 2. Paramètres
# Prix LINÉAIRE : un seul coefficient "par franc"
# Priors supposés (signe attendu), pas tirés de la littérature
# --------------------------------------------------
beta_prix = Parameter("beta_prix", -0.0001)
beta_unicite = Parameter("beta_unicite", 0.0000)
beta_qualite = Parameter("beta_qualite", 0.0001)      # signe négatif, quasi nul
beta_retours_sm = Parameter("beta_retours_sm", 0.0001) # signe positif, quasi nul
beta_retours_neuf = Parameter("beta_retours_neuf", 0.0001)
asc_sm = ASC("asc_sm", 0.0)  # déjà à 0


# --------------------------------------------------
# 3. Fonctions d'utilité
# Le prix entre directement comme un nombre
# --------------------------------------------------
V_sm = (
    asc_sm
    + beta_prix * prix_sm
    + beta_qualite * (qualite_sm == 2)
    + beta_unicite * (unicite_sm == 2)
    + beta_retours_sm * (retours_sm == 2)
)

V_neuf = (
    beta_prix * prix_neuf
    + beta_unicite * (unicite_neuf == 2)
    + beta_retours_neuf * (retours_neuf == 2)
)

V = {
    1: V_sm,
    2: V_neuf
}

candidats:

In [10]:
# --------------------------------------------------
# 4. Préparer les 144 paires candidates
# --------------------------------------------------

colonnes_modele = [
    "Prix_SM",
    "Qualite_SM",
    "Unicite_SM",
    "Retours_SM",
    "Prix_neuf",
    "Unicite_neuf",
    "Retours_neuf"
]

candidates = df_paires_candidates.reset_index(drop=True).copy()


# --------------------------------------------------
# 5. Créer l'évaluateur D-efficient
# --------------------------------------------------

design = EffDesign(
    X=[
        prix_sm,
        qualite_sm,
        unicite_sm,
        retours_sm,
        prix_neuf,
        unicite_neuf,
        retours_neuf
    ],
    ncs=12
)


def evaluer_selection(indices):
    plan = candidates.iloc[indices][colonnes_modele].copy()

    # Calcul de la pénalité si la condition n'est pas respectée
    nb_standard = (plan["Qualite_SM"] == 1).sum()
    penalite = 0.0
    if nb_standard < 5:
        # Pénalité douce : plus il manque de profils standard, plus la D-error sera élevée
        penalite = (5 - nb_standard) * 10.0

    plan.insert(0, "CS", range(1, len(plan) + 1))

    d_error, equilibre = design.evaluate(
        plan,
        V=V,
        criterion="d"
    )

    # On ajoute la pénalité à la D-error trouvée
    return float(d_error) + penalite, float(equilibre)

# --------------------------------------------------
# 6. Recherche locale avec plusieurs départs aléatoires
#    (version limitée en temps : échantillonnage des candidats testés à chaque swap)
# --------------------------------------------------

rng = np.random.default_rng(42)

meilleur_plan = None
meilleur_d_error = np.inf
meilleur_equilibre = None

nombre_depart = 5
iterations_max = 25
taille_echantillon = 70

for depart in range(nombre_depart):

    # tirer un départ qui respecte déjà la règle des 4 "standard"
    while True:
        selection = list(
            rng.choice(len(candidates), size=12, replace=False)
        )
        if (candidates.iloc[selection]["Qualite_SM"] == 1).sum() >= 5:
            break

    d_error, equilibre = evaluer_selection(selection)

    for iteration in range(iterations_max):

        indices_selectionnes = set(selection)
        candidats_restants = [
            i for i in range(len(candidates)) if i not in indices_selectionnes
        ]
        candidats_testes = rng.choice(
            candidats_restants,
            size=min(taille_echantillon, len(candidats_restants)),
            replace=False
        )

        meilleur_echange = None
        meilleur_d_local = d_error
        meilleur_eq_local = equilibre

        for position in range(12):

            for candidate_idx in candidats_testes:

                essai = selection.copy()
                essai[position] = candidate_idx

                d_essai, eq_essai = evaluer_selection(essai)

                if (
                    np.isfinite(d_essai)
                    and d_essai < meilleur_d_local - 1e-10
                ):
                    meilleur_d_local = d_essai
                    meilleur_eq_local = eq_essai
                    meilleur_echange = essai

        if meilleur_echange is None:
            break

        selection = meilleur_echange
        d_error = meilleur_d_local
        equilibre = meilleur_eq_local

    print(f"Départ {depart + 1}/{nombre_depart} terminé — D-error : {d_error:.4f}")

    if d_error < meilleur_d_error:
        meilleur_d_error = d_error
        meilleur_equilibre = equilibre
        meilleur_plan = selection.copy()

# --------------------------------------------------
# 7. Afficher les 12 questions sélectionnées
# --------------------------------------------------

df_questions = candidates.iloc[meilleur_plan].copy()
df_questions = df_questions.reset_index(drop=True)

df_questions.insert(
    0, "Question", range(1, len(df_questions) + 1)
)

print("Nombre de questions :", len(df_questions))
print("D-error final :", meilleur_d_error)
print("Équilibre des utilités (%) :", meilleur_equilibre)

display(df_questions)

Départ 1/5 terminé — D-error : 0.1497
Départ 2/5 terminé — D-error : 0.1497
Départ 3/5 terminé — D-error : 0.1497
Départ 4/5 terminé — D-error : 0.1497
Départ 5/5 terminé — D-error : 0.1497
Nombre de questions : 12
D-error final : 0.14971754922186728
Équilibre des utilités (%) : 99.99951141945495


,Question,Prix_SM,Qualite_SM,Unicite_SM,Retours_SM,Prix_neuf,Unicite_neuf,Retours_neuf
0,1,20,2,1,1,80,2,1
1,2,40,2,1,2,40,2,1
2,3,20,1,2,1,80,1,2
3,4,40,2,2,1,40,1,2
4,5,20,2,1,2,80,2,1
5,6,40,2,1,1,60,2,2
6,7,20,1,1,1,80,2,2
7,8,20,2,1,1,40,2,2
8,9,20,1,1,2,80,2,1
9,10,40,1,1,2,60,2,2


In [11]:
# Dictionnaires de correspondance code -> libellé
labels_qualite = {1: "Bon état, avec traces d'ususre visible", 2: "Comme neuf"}
labels_unicite = {1: "Pièce mainstream", 2: "Trouvaille unique"}
labels_garantie = {1: "Sans retour", 2: "Retours 30j"}

df_questions_lisible = df_questions.copy()

df_questions_lisible["Qualite_SM"] = df_questions_lisible["Qualite_SM"].replace(labels_qualite)
df_questions_lisible["Unicite_SM"] = df_questions_lisible["Unicite_SM"].replace(labels_unicite)
df_questions_lisible["Retours_SM"] = df_questions_lisible["Retours_SM"].replace(labels_garantie)
df_questions_lisible["Unicite_neuf"] = df_questions_lisible["Unicite_neuf"].replace(labels_unicite)
df_questions_lisible["Retours_neuf"] = df_questions_lisible["Retours_neuf"].replace(labels_garantie)

display(df_questions_lisible)

,Question,Prix_SM,Qualite_SM,Unicite_SM,Retours_SM,Prix_neuf,Unicite_neuf,Retours_neuf
0,1,20,Comme neuf,Pièce mainstream,Sans retour,80,Trouvaille unique,Sans retour
1,2,40,Comme neuf,Pièce mainstream,Retours 30j,40,Trouvaille unique,Sans retour
2,3,20,"Bon état, avec traces d'ususre visible",Trouvaille unique,Sans retour,80,Pièce mainstream,Retours 30j
3,4,40,Comme neuf,Trouvaille unique,Sans retour,40,Pièce mainstream,Retours 30j
4,5,20,Comme neuf,Pièce mainstream,Retours 30j,80,Trouvaille unique,Sans retour
5,6,40,Comme neuf,Pièce mainstream,Sans retour,60,Trouvaille unique,Retours 30j
6,7,20,"Bon état, avec traces d'ususre visible",Pièce mainstream,Sans retour,80,Trouvaille unique,Retours 30j
7,8,20,Comme neuf,Pièce mainstream,Sans retour,40,Trouvaille unique,Retours 30j
8,9,20,"Bon état, avec traces d'ususre visible",Pièce mainstream,Retours 30j,80,Trouvaille unique,Sans retour
9,10,40,"Bon état, avec traces d'ususre visible",Pièce mainstream,Retours 30j,60,Trouvaille unique,Retours 30j


**Choix de modélisation retenu**

- Prix linéaire : un seul coefficient par franc (beta_prix). Permet de calculer la disposition à payer (ex. : combien de CHF vaut un vêtement avec un état comme neuf).
- Retours séparés : beta_retours_sm et beta_retours_neuf. Hypothèse : la possibilité de retour rassure davantage pour un pull de seconde main (risque sur l'état, la propreté, la taille) que pour un pull neuf. Séparer les deux permet de tester cette hypothèse statistiquement.
- Unicité des pièces : un seul beta_unicite pour les deux alternatives (pas de raison claire de penser que l'effet diffère).
- Qualité: seulement pour la seconde main (un pull neuf est forcément comme neuf).
- Questions évidentes retirées : 74 paires candidates restantes, toutes avec un vrai arbitrage.


In [12]:
import pandas as pd
from IPython.display import display, HTML

# Parcourir les 12 questions
for idx, row in df_questions_lisible.iterrows():
    num_q = int(row['Question'])

    # Données pour SM et Neuf
    col_sm = [
        f"{row['Prix_SM']} CHF",
        row.get('Qualite_SM'),
        row.get('Unicite_SM'),
        row.get('Retours_SM')
    ]

    col_neuf = [
        f"{row['Prix_neuf']} CHF",
        row.get('Qualite_neuf', 'Neuf'),
        row.get('Unicite_neuf'),
        row.get('Retours_neuf')
    ]

    # Alterner le sens gauche/droite pour l'effet A / B
    inverser = (num_q % 2 == 0)

    if inverser:
        data_paire = {"Neuf": col_neuf, "Seconde Main": col_sm}
    else:
        data_paire = {"Seconde Main": col_sm, "Neuf": col_neuf}

    df_q = pd.DataFrame(data_paire, index=['Prix', 'Qualité', 'Unicité', 'Retours'])

    # Application du style : fond blanc pur, lignes horizontales pour chaque attribut, et trait vertical très net
    styled_df = df_q.style \
        .set_caption(f"<b>Question {num_q} / 12</b>") \
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('text-align', 'left'), ('margin-bottom', '10px'), ('color', '#2c3e50')]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('margin-bottom', '30px'), ('width', '100%'), ('max-width', '600px'), ('font-family', 'sans-serif'), ('background-color', '#ffffff'), ('box-shadow', '0 4px 6px rgba(0,0,0,0.05)')]},
            {'selector': 'th', 'props': [('background-color', '#ffffff'), ('color', '#1e293b'), ('font-weight', '700'), ('font-size', '14px'), ('padding', '10px 14px'), ('text-align', 'center'), ('border-bottom', '2px solid #cbd5e1')]},
            {'selector': 'td', 'props': [('background-color', '#ffffff'), ('color', '#1e293b'), ('padding', '10px 14px'), ('border-bottom', '1px solid #e2e8f0'), ('text-align', 'center'), ('font-size', '13px')]},

            # Colonne des attributs (gauche) : fond blanc et lignes horizontales conservées, avec un trait vertical très net à droite
            {'selector': 'th.row_heading', 'props': [('background-color', '#ffffff'), ('color', '#0f172a'), ('font-weight', '600'), ('width', '100px'), ('text-align', 'left'), ('border-right', '3px solid #1e293b'), ('border-bottom', '2px solid #cbd5e1')]},
            {'selector': 'td.row_heading', 'props': [('background-color', '#ffffff'), ('color', '#0f172a'), ('font-weight', '600'), ('text-align', 'left'), ('border-right', '3px solid #1e293b'), ('border-bottom', '1px solid #e2e8f0')]},

            # Fermer le bloc tout en haut à gauche
            {'selector': 'thead tr th:first-child', 'props': [('border-right', '3px solid #1e293b'), ('background-color', '#ffffff')]},

            # Ligne de séparation entre l'Alternative A et l'Alternative B (milieu)
            {'selector': 'td:nth-child(2), th:nth-child(2)', 'props': [('border-right', '2px solid #cbd5e1')]}
        ])

    display(styled_df)

,Seconde Main,Neuf
Prix,20 CHF,80 CHF
Qualité,Comme neuf,Neuf
Unicité,Pièce mainstream,Trouvaille unique
Retours,Sans retour,Sans retour


,Neuf,Seconde Main
Prix,40 CHF,40 CHF
Qualité,Neuf,Comme neuf
Unicité,Trouvaille unique,Pièce mainstream
Retours,Sans retour,Retours 30j


,Seconde Main,Neuf
Prix,20 CHF,80 CHF
Qualité,"Bon état, avec traces d'ususre visible",Neuf
Unicité,Trouvaille unique,Pièce mainstream
Retours,Sans retour,Retours 30j


,Neuf,Seconde Main
Prix,40 CHF,40 CHF
Qualité,Neuf,Comme neuf
Unicité,Pièce mainstream,Trouvaille unique
Retours,Retours 30j,Sans retour


,Seconde Main,Neuf
Prix,20 CHF,80 CHF
Qualité,Comme neuf,Neuf
Unicité,Pièce mainstream,Trouvaille unique
Retours,Retours 30j,Sans retour


,Neuf,Seconde Main
Prix,60 CHF,40 CHF
Qualité,Neuf,Comme neuf
Unicité,Trouvaille unique,Pièce mainstream
Retours,Retours 30j,Sans retour


... (2 outputs left out)

,Seconde Main,Neuf
Prix,20 CHF,80 CHF
Qualité,"Bon état, avec traces d'ususre visible",Neuf
Unicité,Pièce mainstream,Trouvaille unique
Retours,Retours 30j,Sans retour


,Neuf,Seconde Main
Prix,60 CHF,40 CHF
Qualité,Neuf,"Bon état, avec traces d'ususre visible"
Unicité,Trouvaille unique,Pièce mainstream
Retours,Retours 30j,Retours 30j


,Seconde Main,Neuf
Prix,20 CHF,80 CHF
Qualité,Comme neuf,Neuf
Unicité,Trouvaille unique,Pièce mainstream
Retours,Sans retour,Retours 30j


,Neuf,Seconde Main
Prix,40 CHF,20 CHF
Qualité,Neuf,"Bon état, avec traces d'ususre visible"
Unicité,Trouvaille unique,Pièce mainstream
Retours,Retours 30j,Retours 30j


question piège

In [13]:
# 1. Création de la question piège
question_piege = pd.DataFrame([{
    'Question': 99,  # Numéro distinct pour l'identifier facilement
    'Prix_SM': 60,
    'Qualite_SM': "Bon état, avec traces d'ususre visible",
    'Unicite_SM': "Pièce mainstream",
    'Retours_SM': "Sans retour",
    'Prix_neuf': 40,
    'Qualite_neuf': "Neuf",
    'Unicite_neuf': "Trouvaille unique",
    'Retours_neuf': "Retours 30j"
}])

# 2. Préparation des données pour l'affichage
row = question_piege.iloc[0]
num_q = int(row['Question'])

col_sm = [
    f"{row['Prix_SM']} CHF",
    row.get('Qualite_SM'),
    row.get('Unicite_SM'),
    row.get('Retours_SM')
]

col_neuf = [
    f"{row['Prix_neuf']} CHF",
    row.get('Qualite_neuf', 'Neuf'),
    row.get('Unicite_neuf'),
    row.get('Retours_neuf')
]

# Alternative A : Seconde Main / Alternative B : Neuf
data_paire = {
    "Seconde Main": col_sm,
    "Neuf": col_neuf
}

df_q = pd.DataFrame(data_paire, index=['Prix', 'Qualité', 'Unicité', 'Retours'])

# 3. Affichage avec votre design validé
styled_df = df_q.style \
    .set_caption("<b>Question Piège</b> <span style='color: #e74c3c; font-weight: normal;'>(Choix évident : Neuf)</span>") \
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('text-align', 'left'), ('margin-bottom', '10px'), ('color', '#2c3e50')]},
        {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('margin-bottom', '30px'), ('width', '100%'), ('max-width', '600px'), ('font-family', 'sans-serif'), ('background-color', '#ffffff'), ('box-shadow', '0 4px 6px rgba(0,0,0,0.05)')]},
        {'selector': 'th', 'props': [('background-color', '#ffffff'), ('color', '#1e293b'), ('font-weight', '700'), ('font-size', '14px'), ('padding', '10px 14px'), ('text-align', 'center'), ('border-bottom', '2px solid #cbd5e1')]},
        {'selector': 'td', 'props': [('background-color', '#ffffff'), ('color', '#1e293b'), ('padding', '10px 14px'), ('border-bottom', '1px solid #e2e8f0'), ('text-align', 'center'), ('font-size', '13px')]},

        # Colonne des attributs (gauche) : fond blanc, lignes horizontales, trait vertical très net à droite
        {'selector': 'th.row_heading', 'props': [('background-color', '#ffffff'), ('color', '#0f172a'), ('font-weight', '600'), ('width', '100px'), ('text-align', 'left'), ('border-right', '3px solid #1e293b'), ('border-bottom', '2px solid #cbd5e1')]},
        {'selector': 'td.row_heading', 'props': [('background-color', '#ffffff'), ('color', '#0f172a'), ('font-weight', '600'), ('text-align', 'left'), ('border-right', '3px solid #1e293b'), ('border-bottom', '1px solid #e2e8f0')]},

        # Fermer le bloc tout en haut à gauche
        {'selector': 'thead tr th:first-child', 'props': [('border-right', '3px solid #1e293b'), ('background-color', '#ffffff')]},

        # Ligne de séparation entre l'Alternative A et l'Alternative B (milieu)
        {'selector': 'td:nth-child(2), th:nth-child(2)', 'props': [('border-right', '2px solid #cbd5e1')]}
    ])

display(styled_df)

,Seconde Main,Neuf
Prix,60 CHF,40 CHF
Qualité,"Bon état, avec traces d'ususre visible",Neuf
Unicité,Pièce mainstream,Trouvaille unique
Retours,Sans retour,Retours 30j


**Beta pre-test**

In [14]:
from google.colab import files
uploaded = files.upload()

Saving Étude sur les choix de consommation _ La mode de seconde main face à la Fast Fashion._Submissions_2026-11-10.csv to Étude sur les choix de consommation _ La mode de seconde main face à la Fast Fashion._Submissions_2026-11-10.csv


import your file:

In [15]:
import io
# Charger le CSV dans un DataFrame
filename = next(iter(uploaded))
df_answer = pd.read_csv(io.BytesIO(uploaded[filename]))

# Afficher les premières lignes
display(df_answer.head(15))

,Submission ID,Respondent ID,Submitted at,Quel âge avez-vous ?,À quel genre vous identifiez-vous ?,Où résidez-vous principalement ?,"Actuellement, vous êtes... ?",Achetez-vous vous-même vos vêtements ?,À quelle fréquence achetez-vous de nouveaux vêtements (neufs ou d'occasion) ?,"De combien d'argent disposez-vous chaque mois après vos dépenses fixes (loyer, assurances, transports) ?",...,Situation 9 sur 13 : quel pull choisiriez-vous ?,Situation 10 sur 13 : quel pull choisiriez-vous ?,Situation 11 sur 13 : quel pull choisiriez-vous ?,Situation 12 sur 13 : quel pull choisiriez-vous ?,Situation 13 sur 13 : quel pull choisiriez-vous ?,Je suis préoccupé(e) par l’écologie en général.,Je prends en compte l’impact environnemental quand j’achète des vêtements.,Je suis prêt(e) à faire des efforts pour réduire ma consommation de vêtements neufs.,Mon style vestimentaire est un moyen d’exprimer ma personnalité.,Je recherche des pièces originales ou introuvables en magasin classique.
0,9NjbZzG,QKGJ7dl,2026-11-07 13:54:16,25-34 ans,Homme,Suisse romande,Etudiant,Oui,Quelques fois par an,200 à 499 .-,...,Neuf,Seconde Main,Neuf,Seconde Main,Neuf,7.0,4.0,6.0,7.0,6.0
1,o9LXMV5,VLr4J4a,2026-11-07 14:04:42,25-34 ans,Femme,Suisse romande,Etudiant,Oui,Tous les mois,200 à 499 .-,...,Neuf,Neuf,Neuf,Seconde Main,Neuf,10.0,7.0,10.0,10.0,10.0
2,VpO2Vol,EkpN1KL,2026-11-07 14:24:11,18-24 ans,Femme,Suisse romande,Etudiant,Oui,Quelques fois par an,200 à 499 .-,...,Neuf,Seconde Main,Neuf,Seconde Main,Neuf,3.0,2.0,0.0,8.0,6.0
3,NqZ5K6b,pbg7K2P,2026-11-08 07:43:09,35+,Homme,Suisse romande,Jeune actif,Non,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,aOMNbxW,pbg7K2P,2026-11-08 07:43:41,35+,Homme,Suisse romande,Jeune actif,Oui,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,9NKqqP5,RGOj9lQ,2026-11-08 08:01:33,25-34 ans,Femme,Suisse romande,Etudiant,Oui,Tous les mois,200 à 499 .-,...,Neuf,Seconde Main,Seconde Main,Seconde Main,Neuf,8.0,3.0,7.0,7.0,6.0
6,laEbdQW,BzvePZ1,2026-11-08 09:05:55,25-34 ans,Femme,Suisse romande,Jeune actif,Oui,Quelques fois par an,200 à 499 .-,...,Seconde Main,Seconde Main,Neuf,Seconde Main,Neuf,5.0,3.0,5.0,8.0,8.0
7,Vp6pVWa,Bzv9qMR,2026-11-08 14:08:56,25-34 ans,Femme,Suisse romande,Jeune actif,Oui,Quelques fois par an,200 à 499 .-,...,Neuf,Seconde Main,Neuf,Seconde Main,Neuf,7.0,5.0,6.0,9.0,8.0
8,jeVoA6Q,WO4dbLL,2026-11-09 09:15:54,18-24 ans,Homme,Suisse romande,Etudiant,Oui,Tous les mois,200 à 499 .-,...,Neuf,Seconde Main,Neuf,Seconde Main,Neuf,8.0,4.0,6.0,5.0,7.0
9,2j6KYWV,rjX1rrN,2026-11-09 12:20:48,25-34 ans,Homme,Suisse romande,Etudiant,Oui,Tous les mois,500 à 1000 .-,...,Seconde Main,Seconde Main,Seconde Main,Seconde Main,Seconde Main,10.0,7.0,10.0,10.0,10.0


In [16]:
# Suppression des répondants "35+" et de la colonne inutile 'Submission ID'
if "Submission ID" in df_answer.columns:
    df_answer = df_answer.drop(columns=["Submission ID"])

if "Quel âge avez-vous ?" in df_answer.columns:
    df_answer = df_answer[df_answer["Quel âge avez-vous ?"] != "35+"].copy()

# ---> FILTRE DE LA QUESTION PIÈGE AU TOUT DÉBUT <---
col_piege = "Situation 8 sur 13 : quel pull choisiriez-vous ?"
if col_piege in df_answer.columns:
    nb_avant = len(df_answer)
    # On ne garde que ceux qui ont répondu "Neuf" (la bonne réponse évidente)
    df_answer = df_answer[df_answer[col_piege] == "Neuf"].copy()
    nb_apres = len(df_answer)
    print(f"Filtre question piège : {nb_avant - nb_apres} répondant(s) écarté(s) pour mauvaise réponse.")

# 2. Dictionnaires de mapping global pour les recodages catégoriels
mappings = {
    "Quel âge avez-vous ?": {"18-24 ans": 1, "25-34 ans": 2},
    "À quel genre vous identifiez-vous ?": {"Homme": 1, "Femme": 2},
    "Actuellement, vous êtes... ?": {"Etudiant": 1, "Jeune actif": 2},
    "Où résidez-vous principalement ?": {"Suisse romande": 1, "Suisse alémanique": 2, "Suisse italienne": 3, "A l'étranger": 4},
    "Achetez-vous vous-même vos vêtements ?": {"Oui": 1, "Non": 2},
    "À quelle fréquence achetez-vous de nouveaux vêtements (neufs ou d'occasion) ?": {"Toutes les semaines": 1, "Tous les mois": 2, "Quelques fois par an": 3},
    "De combien d'argent disposez-vous chaque mois après vos dépenses fixes (loyer, assurances, transports) ?": {"moins de 200.-": 1, "200 à 499 .-": 2, "500 à 1000 .-": 3, "plus de 1000.-": 4},
    "Combien dépenser vous en moyenne par mois pour vos vêtements ?": {"moins de 50.-": 1, "50 à 99 .-": 2, "100 à 200.-": 3, "plus de 200.-": 4},
    "Avez-vous acheté des vêtements de seconde main au cours des 12 derniers mois ?": {"Oui": 1, "Non": 2},
    "Où avez-vous fait votre achat ?": {"Plateforme en ligne": 1, "Magasin": 2}
}

# Application propre des mappings existants dans le DataFrame
for col, mapping in mappings.items():
    if col in df_answer.columns:
        df_answer[col] = df_answer[col].replace(mapping)

# 3. Traitement vectorisé des colonnes booléennes (raisons de non-achat)
for col in df_answer.columns:
    if "Pour quelle(s) raison(s) n’avez-vous pas acheté" in col and col != "Pour quelle(s) raison(s) n’avez-vous pas acheté de vêtements de seconde main ?":
        df_answer[col] = df_answer[col].replace({True: 1, False: 0}).infer_objects(copy=False).fillna(0).astype(int)

print("Nettoyage et encodage optimisés avec succès ! Dimensions finales de df_answer :", df_answer.shape)

Filtre question piège : 1 répondant(s) écarté(s) pour mauvaise réponse.
Nettoyage et encodage optimisés avec succès ! Dimensions finales de df_answer : (12, 40)


/tmp/ipykernel_375/572269071.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_answer[col] = df_answer[col].replace(mapping)
/tmp/ipykernel_375/572269071.py:39: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_answer[col] = df_answer[col].replace({True: 1, False: 0}).infer_objects(copy=False).fillna(0).astype(int)
/tmp/ipykernel_375/572269071.py:39: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=Fal

préparation question choix

In [17]:
# 1. Dictionnaire de correspondance (gérant le décalage de la question piège n°8)
correspondance_situations = {
    "Situation 1 sur 13 : quel pull choisiriez-vous ?": 1,
    "Situation 2 sur 13 : quel pull choisiriez-vous ?": 2,
    "Situation 3 sur 13 : quel pull choisiriez-vous ?": 3,
    "Situation 4 sur 13 : quel pull choisiriez-vous ?": 4,
    "Situation 5 sur 13 : quel pull choisiriez-vous ?": 5,
    "Situation 6 sur 13 : quel pull choisiriez-vous ?": 6,
    "Situation 7 sur 13 : quel pull choisiriez-vous ?": 7,
    "Situation 8 sur 13 : quel pull choisiriez-vous ?": None,  # <- QUESTION PIÈGE (ignorée pour Biogeme)
    "Situation 9 sur 13 : quel pull choisiriez-vous ?": 8,
    "Situation 10 sur 13 : quel pull choisiriez-vous ?": 9,
    "Situation 11 sur 13 : quel pull choisiriez-vous ?": 10,
    "Situation 12 sur 13 : quel pull choisiriez-vous ?": 11,
    "Situation 13 sur 13 : quel pull choisiriez-vous ?": 12,
}

# 2. Transformation du format Large (df_answer) vers Long (df_beta pour Biogeme)
records = []

for idx, respondent in df_answer.iterrows():
    resp_id = respondent["Respondent ID"]

    for col_name, q_num in correspondance_situations.items():
        if q_num is None:
            continue  # Ignore la question piège (Situation 8)

        choix_str = respondent.get(col_name)
        if pd.isna(choix_str):
            continue  # Ignore si pas de réponse

        # Codage Biogeme : 1 = Seconde Main, 2 = Neuf
        choice_val = 1 if str(choix_str).strip() == "Seconde Main" else 2

        # Récupération des attributs depuis df_questions
        row_design = df_questions[df_questions["Question"] == q_num].iloc[0]

        records.append({
            "Respondent_ID": resp_id,
            "Task_ID": q_num,
            "CHOICE": choice_val,
            "Prix_SM": row_design["Prix_SM"],
            "Qualite_SM": row_design["Qualite_SM"],
            "Unicite_SM": row_design["Unicite_SM"],
            "Retours_SM": row_design["Retours_SM"],
            "Prix_neuf": row_design["Prix_neuf"],
            "Unicite_neuf": row_design["Unicite_neuf"],
            "Retours_neuf": row_design["Retours_neuf"],
            # Variables individuelles utiles pour les interactions
            "Age": respondent.get("Quel âge avez-vous ?"),
            "Genre": respondent.get("À quel genre vous identifiez-vous ?")
        })

df_beta = pd.DataFrame(records)
print("Base df_beta prête pour Biogeme ! Dimensions :", df_beta.shape)
display(df_beta.head(10))

Base df_beta prête pour Biogeme ! Dimensions : (144, 12)


,Respondent_ID,Task_ID,CHOICE,Prix_SM,Qualite_SM,Unicite_SM,Retours_SM,Prix_neuf,Unicite_neuf,Retours_neuf,Age,Genre
0,QKGJ7dl,1,2,20,2,1,1,80,2,1,2,1
1,QKGJ7dl,2,2,40,2,1,2,40,2,1,2,1
2,QKGJ7dl,3,1,20,1,2,1,80,1,2,2,1
3,QKGJ7dl,4,1,40,2,2,1,40,1,2,2,1
4,QKGJ7dl,5,1,20,2,1,2,80,2,1,2,1
5,QKGJ7dl,6,2,40,2,1,1,60,2,2,2,1
6,QKGJ7dl,7,2,20,1,1,1,80,2,2,2,1
7,QKGJ7dl,8,2,20,2,1,1,40,2,2,2,1
8,QKGJ7dl,9,1,20,1,1,2,80,2,1,2,1
9,QKGJ7dl,10,2,40,1,1,2,60,2,2,2,1


**BIOGEME**
final questions:

In [18]:
!pip install biogeme

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.8/863.8 kB 9.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 29.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 64.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 55.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 90.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 649.8/649.8 kB 22.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 85.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.6/619.6 kB 17.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.6 MB/s  0:00:00
  Attempting uninstall: traitlets
    Found existing installation: traitlets 5.7.1
    Uninstalling traitlets-5.7.1:
      Successfully uninstalled traitlets-5.7.1
  Attem

In [19]:
import biogeme.database as db
import biogeme.biogeme as bio
from biogeme.expressions import Beta, Variable
import biogeme.models as models

/usr/local/lib/python3.13/dist-packages/tqdm_joblib/__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [20]:
# Garder une table de correspondance (utile si tu dois retrouver un répondant)
df_beta["Respondent_ID_str"] = df_beta["Respondent_ID"]
df_beta["ID"], id_mapping = pd.factorize(df_beta["Respondent_ID"])
df_beta["ID"] = df_beta["ID"] + 1  # pour éviter un ID = 0

# Colonne numérique à utiliser dans Biogeme, on peut retirer la colonne texte
df_biogeme = df_beta.drop(columns=["Respondent_ID", "Respondent_ID_str"])

# Vérifications avant de passer à Biogeme
print(df_biogeme.dtypes)
print(df_biogeme.isna().sum())
print(df_biogeme["ID"].value_counts())  # chaque ID doit avoir 12 lignes

from biogeme.database import Database
database = Database("pretest", df_biogeme)
database.panel("ID")  # active la structure panel (répétitions par individu)

Task_ID         int64
CHOICE          int64
Prix_SM         int64
Qualite_SM      int64
Unicite_SM      int64
Retours_SM      int64
Prix_neuf       int64
Unicite_neuf    int64
Retours_neuf    int64
Age             int64
Genre           int64
ID              int64
dtype: object
Task_ID         0
CHOICE          0
Prix_SM         0
Qualite_SM      0
Unicite_SM      0
Retours_SM      0
Prix_neuf       0
Unicite_neuf    0
Retours_neuf    0
Age             0
Genre           0
ID              0
dtype: int64
ID
9     24
2     12
1     12
3     12
4     12
6     12
5     12
7     12
8     12
10    12
11    12
Name: count, dtype: int64
